In [ ]:
import os
import json
import numpy as np

DIR_LIST = [
    [
        "./SynAgent0/outputs/responses_SynAgent0_0",
        "./SynAgent0/outputs/responses_SynAgent0_1",
        "./SynAgent0/outputs/responses_SynAgent0_2",
        "./SynAgent0/outputs/responses_SynAgent0_3",
        "./SynAgent0/outputs/responses_SynAgent0_4",
        "./SynAgent0/outputs/responses_SynAgent0_5",
    ],
    [
        "./SynAgent1/outputs/responses_SynAgent1_0",
        "./SynAgent1/outputs/responses_SynAgent1_1",
        "./SynAgent1/outputs/responses_SynAgent1_2",
        "./SynAgent1/outputs/responses_SynAgent1_3",
        "./SynAgent1/outputs/responses_SynAgent1_4",
        "./SynAgent1/outputs/responses_SynAgent1_5",
    ],
    [
        "./SynAgent2/outputs/responses_SynAgent2_0",
        "./SynAgent2/outputs/responses_SynAgent2_1",
        "./SynAgent2/outputs/responses_SynAgent2_2",
        "./SynAgent2/outputs/responses_SynAgent2_3",
        "./SynAgent2/outputs/responses_SynAgent2_4",
        "./SynAgent2/outputs/responses_SynAgent2_5",
    ]
]


def robust_min_max_normalize(arr):
    finite_values = arr[np.isfinite(arr)]
    x_min = np.min(finite_values) if finite_values.size > 0 else 0
    x_max = np.max(finite_values) if finite_values.size > 0 else 0
    
    if x_max == x_min:
        return np.zeros_like(arr)
    
    normalized = np.where(
        np.isneginf(arr), -np.inf, (arr - x_min) / (x_max - x_min)
    )
    return normalized


def get_reply_lengths(directory):
    reply_lengths = list()
    folder_name = os.path.basename(directory)
    for i in range(18):
        file_path = f"{directory}/{folder_name}_{i}.json"
        with open(file_path, "r", encoding="utf-8") as f:
            replies = json.load(f)
            for reply in replies:
                if reply is None:
                    reply_lengths.append(-1)
                else:
                    reply_lengths.append(len(reply))
    return reply_lengths


def gather(directory):
    folder_name = os.path.basename(directory)
    reply_lengths = get_reply_lengths(directory)
    records = []
    with open(f"{directory}/evaluation1-{folder_name}.json", "r", encoding="utf-8") as f1:
        evaluation1 = json.load(f1)
    with open(f"{directory}/evaluation2-{folder_name}.json", "r", encoding="utf-8") as f2:
        evaluation2 = json.load(f2)
    with open(f"{directory}/evaluation3_v1check-{folder_name}.json", "r", encoding="utf-8") as f3_v1:
        evaluation3_v1 = json.load(f3_v1)
    with open(f"{directory}/evaluation3_v2check-{folder_name}.json", "r", encoding="utf-8") as f3_v2:
        evaluation3_v2 = json.load(f3_v2)
        
    for entry1, entry2, entry3_1, entry3_2, len_reply in zip(
        evaluation1, evaluation2, evaluation3_v1, evaluation3_v2, reply_lengths
    ):
        if not (entry1["id"] == entry2["id"] and entry2["id"] == entry3_1["id"] and entry3_1["id"] == entry3_2["id"]):
            print("Exception: Diffierent index.")
            
        if entry1["correctness"] != 1:
            records.append({"id": entry1["id"], "correct": False, "rationale_ratio": None, "validity0": None, "validity": None})
        elif entry2["rationale"] is None:
            records.append({"id": entry1["id"], "correct": True, "rationale_ratio": None, "validity0": None, "validity": None})
        else:
            validity0 = None if entry3_1["ppl"] is None else (1 / entry3_1["ppl"]["ppl_value"] * (int(entry3_1["ppl"]["correctness"] == 1) * 2 - 1))
            validity = None if entry3_2["ppl"] is None else (1 / entry3_2["ppl"]["ppl_value"] * (int(entry3_2["ppl"]["correctness"] == 1) * 2 - 1))
            records.append({
                "id": entry1["id"],
                "correct": True,
                "rationale_ratio": len(entry2["rationale"]) / len_reply,
                "validity0": (validity0, entry3_1["ppl"]["correctness"]) if validity0 is not None else None,
                "validity": (validity, entry3_2["ppl"]["correctness"]) if validity is not None else None
            })
    return records

records_list = []
for directorys in DIR_LIST:
    for directory in directorys:
        records = gather(directory)
        records_list.append(records)


In [ ]:
dict_selected = dict()
for index, item in enumerate(zip(*records_list)):
    if False in [record['id'] == item[0]['id'] for record in item]:
        print("Exception: Diffierent index.")

    count_correct = np.array([
        sum([int(item[k]["correct"]) for k in range(6)]),
        sum([int(item[k + 6]["correct"]) for k in range(6)]),
        sum([int(item[k + 12]["correct"]) for k in range(6)]),
    ])
    count_valid = np.array([
        sum([1 if (item[k]["validity"] is not None and item[k]["validity"][1] == 1) else 0 for k in range(6)]),
        sum([1 if (item[k + 6]["validity"] is not None and item[k + 6]["validity"][1] == 1) else 0 for k in range(6)]),
        sum([1 if (item[k + 12]["validity"] is not None and item[k + 12]["validity"][1] == 1) else 0 for k in range(6)]),
    ])
    grp = np.argmax(100 * count_valid + count_correct)
    

    item = item[grp * 6: (grp + 1) * 6]
    arr_correct = np.array(
        [0. if record["correct"] else float('-inf') for record in item]
    )
    arr_beneficial = np.array(
        [0 if (record["validity"] is not None and record["validity"][1] == 1) else -10000 for record in item]
    )
    arr_rnaler = np.array(
        [record["rationale_ratio"] if record["rationale_ratio"] is not None else float('-inf') for record in item]
    )
    arr_valid = np.array(
        [record["validity"][0] if record["validity"] is not None else float('-inf') for record in item]
    )

    arr_rnaler = np.where(arr_rnaler <= 1.2, arr_rnaler, 1.2)
    arr_score = arr_correct + arr_beneficial + 0.5 * arr_rnaler + 0.5 * (arr_valid + 1) / 2
    loc = np.argmax(arr_score)
    
    if np.all(np.isneginf(arr_correct)) or np.all(np.isneginf(arr_rnaler)) or np.all(np.isneginf(arr_valid)):
        dict_selected[index] = -1
    else:
        dict_selected[index] = loc + grp * 6

In [ ]:
from PIL import Image
from datasets import load_dataset

DATASET_PATH = "./DatasetDownload/my_dataset_cache/dataset_0"
DATASET_SPLIT = "train"
NUM_SHARDS = 18
dataset = load_dataset(DATASET_PATH, split=DATASET_SPLIT)

replies_list = [
    [[], [], [], [], [], []],
    [[], [], [], [], [], []],
    [[], [], [], [], [], []],
]
for i in range(3 * 6):
    path_dir = DIR_LIST[i // 6][i % 6]
    for j in range(NUM_SHARDS):
        path_file = f"{path_dir}/{os.path.basename(path_dir)}_{j}.json"
        with open(path_file, "r", encoding="utf-8") as f:
            replies_list[i // 6][i % 6].extend(json.load(f)) # ?


In [ ]:
cnt = 0
for idx, datum in enumerate(dataset):    
    if idx in dict_selected.keys():
        loc = dict_selected[idx]
        if loc < 0 or loc > 17:
            print("Exception: loc < 0  or loc > 17 .")
            break
        conversations = datum['conversations']
        question = conversations[0]['value']
        grd_truth = conversations[1]['value']

        image_path = f"{idx}.jpg"
        image = Image.open(f'./DatasetDownload/my_dataset_cache/dataset_1/images/{image_path}')

        print("##################################################################")
        print("---------- question ----------")
        print(question)
        print("---------- image ----------")
        image.show()
        print("---------- ground truth ----------")
        print(grd_truth)
        print("---------- reply ----------")
        print(replies_list[loc // 6][loc % 6][idx])

        cnt += 1
        if cnt > 50:
            break
    


In [ ]:
def extract_answer(reply: str):
    if '</think>' in reply and '<answer>' in reply:
        cot = f"{reply.split('</think>')[0]}</think>"
        answer = f"<answer>{reply.split('<answer>')[1]}"
    elif '</think>' in reply and '<answer>' not in reply:
        cot = f"{reply.split('</think>')[0]}</think>"
        answer = f"<answer>{reply.split('</think>')[1]}</answer>" if reply.split('</think>')[1] != '' else None
    elif  '</think>' not in reply and '<answer>' in reply:
        cot = f"{reply.split('<answer>')[0]}</think>"
        answer = f"<answer>{reply.split('<answer>')[1]}"
    else:
        cot = None
        answer = None
    return cot, answer

In [ ]:
def generate_answer(cot: str):
    if '**Final Answer:**' in cot[-200: ]:
        answer = cot[-200: ].split('**Final Answer:**')[-1].split('</think>')[0]
        answer = f"<answer>**Answer:** {answer}</answer>"
    elif '**Answer:**' in cot[-200: ]:
        answer = cot[-200: ].split('**Answer:**')[-1].split('</think>')[0]
        answer = f"<answer>**Answer:** {answer}</answer>"
    elif 'Answer: ' in cot[-200: ]:
        answer = cot[-200: ].split('Answer: ')[-1].split('</think>')[0]
        answer = f"<answer>**Answer:** {answer}</answer>"
    elif 'answer' in cot[-100: ]:
        answer = cot[-100: ].split('answer')[-1].split('</think>')[0]
        answer = f"<answer>Answer {answer}</answer>"
    else:
        answer = cot.split('\n')[-2]
        answer = f"<answer>{answer}</answer>"
    return answer

In [ ]:
import base64
import os

def save_base64_to_jpg(base64_data: str, output_path: str, overwrite: bool = False):
    try:
        if base64_data.startswith('data:'):
            base64_data = base64_data.split(',', 1)[1]
        
        image_data = base64.b64decode(base64_data)
        
        output_dir = os.path.dirname(output_path)
        if output_dir and not os.path.exists(output_dir):
            os.makedirs(output_dir)
        
        if os.path.exists(output_path) and not overwrite:
            print(f"File already exists: {output_path}，skipping saving")
            return None
        
        with open(output_path, 'wb') as f:
            f.write(image_data)
        return output_path
    
    except Exception as e:
        print(f"Error occured when processing the image: {str(e)}")
        return None


In [ ]:
from tqdm import tqdm

json_list = list()
for idx, datum in tqdm(enumerate(dataset)):   
    if idx in dict_selected.keys():
        conversations = datum['conversations']
        question = conversations[0]['value']
        question = question if '<image>' in question else "<image>\n" + question
        image_base64 = datum['image']

        loc = dict_selected[idx]
        if loc < 0 or loc > 17:
            print("Exception: loc < 0 or loc > 17 .")
            continue
        else:
            reply = replies_list[loc // 6][loc % 6][idx]
            cot, answer = extract_answer(reply)
            if cot is None:
                continue
            elif answer is None:
                answer = generate_answer(cot)
            
        image_path = f"{idx}.jpg"
        json_list.append({
            'id': datum['id'],
            'image': image_path,
            'conversations': [
                {
                    'from': 'human', 
                    'value': question
                }, 
                {
                    'from': 'assistant', 
                    'value': f"{cot} {answer}"
                }
            ]
        })

with open(
    "./DatasetDownload/my_dataset_cache/dataset_1_v2/data.json",
    mode="w", encoding="utf-8"
) as f:
    json.dump(json_list, f, ensure_ascii=False, indent=2)


In [ ]:
import json

with open(
    "./DatasetDownload/my_dataset_cache/dataset_1_v2/data.json",
    mode="r", encoding="utf-8"
) as f:
    json_list = json.load(f)


In [ ]:
for datum in json_list:
    question = datum['conversations'][0]['value']
    reply = datum['conversations'][1]['value']
    if question.count('<image>') != 1:
        print("Exception: <image>; question", datum['id'])
    if question.count('<video>') > 0:
        print("Exception: <video>; question", datum['id'])
    if reply.count('<image>') > 0:
        print("Exception: <image>; reply", datum['id'])
    if reply.count('<video>') > 0:
        print("Exception: <video>; reply", datum['id'])

In [ ]:
import os
media_path = "./DatasetDownload/my_dataset_cache/dataset_1_v2/images"
l = os.listdir(media_path)
print(len(l))
print(len(json_list))

In [ ]:
from PIL import Image

for idx, datum in enumerate(json_list):
    conversations = datum['conversations']
    question = conversations[0]['value']
    response = conversations[1]['value']

    if (idx + 1) in [34800, 34801, 34802, 34803, 34804, 34805, 34806, 34807, 34808, 34809, 34810]:

        image_path = f"{media_path}/{datum['image']}"
        image = Image.open(image_path)
        
        print(f"##### {idx + 1} #####")
        print("---------- question ----------")
        print(question)
        print("---------- response ----------")
        print(response)
        print("---------- image ----------")
        image.show()
        # break